# Upsampling

**Learning goals:**
* Experiencing the effects of interpolation and decimation on the frequency content of a signal
* Understanding the role of filters in sample rate conversion

**Relevant reading:**
* Chapters 10.1 - 10.6 in the book "Understanding Digital Signal Processing"

**Libraries and notebook config:**

In [ ]:
from scipy.io import wavfile
import scipy.signal as sig
import numpy as np
from IPython.display import Audio
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [8.00, 4.5]
plt.rcParams["figure.autolayout"] = True
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.xmargin"] = 0.0

%matplotlib inline

### Problem description

In this problem we will increase the sample rate of the signal $x[n]$ recorded in the file `task1_signal.mat`. Run the code cell below to load the signal from the file, and plot the magnitude spectrum of $x[n]$.

In [ ]:
from scipy.io import loadmat
data = loadmat("data/task1_signal.mat", squeeze_me=True)
x_n = data["x_n"]
fs = data["fs"]

plt.magnitude_spectrum(x_n, Fs=fs);
plt.title("Magnitude spectrum of signal stored in 'task1_signal.mat'")
plt.show()

### Upsampling

The next component of sample rate conversion we will explore is Upsampling; the process of increasing the sample rate of a discrete signal $x_{\text{old}}[n]$ by a factor of $L$. Described mathematically, the upsampled signal $x_{\text{new}}[m]$ is a discrete-time signal where the new sampling frequency is 
$$f_{s, \text{ new}} = f_{s, \text{ old}} \cdot L$$

For upsampling to be successful, we need to estimate the value of the original analog signal $x(t)$ at points *between* the sampling times $t_n \in \{0, T_s, 2T_s, \ldots\}$. The term for this is [*interpolation*](https://en.wikipedia.org/wiki/Interpolation), and there exists many approaches to interpolation depending on the field.

From a signal processing perspective, the goal can be defined as follows: find the discrete-time signal $x_{\text{new}}[m]$ whose frequency-domain representation $X_{\text{new}}(\hat{\omega})$ is:

$$X_{\text{new}}(\hat{\omega}) \begin{cases} X_{\text{old}}(\hat{2\omega}), & 0 \leq \hat{\omega} \leq \frac{\pi}{2} \\ 0, & \frac{\pi}{2}  < \hat{\omega} \leq \pi \end{cases}$$


This is accomplished in two stages:
1. Interpolate the signal with $L-1$ zeroes between each sample (sometimes referred to as "zero stuffing"). This will create a signal with high-frequency spectral copies of $x(t)$
2. Use a lowpass filter to remove the spectral copies, as well as amplify the frequencies in the frequency range $0 \leq f < \frac{f_{s, \text{ old}}}{2}$.

![](figures/upsampling.png)

Let's take a closer look at the first stage, and what the frequency domain of a signal looks like if we interpolate by a factor of $L=3$...

## a)
* Use python to interpolate the signal $x[n]$ by a factor of $3$. The output should be a new array `x_new` with twice the lenght of `x_n`, where each original sample value is interspersed with a zero ($0$). Show the magnitude spectrum of both the original signal $x[n]$, and the upsampled signal $x_{\text{new}}[m]$.

In [ ]:
# WRITE YOUR CODE IN THIS CELL:

[Here](figures/task2a_lf.png) is a plot showing what the interpolated signal magnitude spectrum should look like. 

## b)

* Study the magnitude spectrum in problem **a)**, and answer the following:
    * What do you suppose is the connection between the shape of the high-frequency spectral image, and the baseband signal?
    * Why is the magnitude in the interpolated signal now $\frac{1}{3}$ of the original magnitude?


<div class="alert alert-info">
<h4> Answer theory questions here!</h4>
</div>



Our goal with upsampling the signal $x[n]$ is to create a new signal $x_{\text{new}}[m]$ whose magnitude spectrum equals that of $x[n]$ for $f \in (0\text{Hz}, 50\text{Hz})$ and equals zero for the frequencies above $\frac{f_{s, \text{ old}}}{2}$. This is done by filtering the "zero stuffed" signal from problem **a)** with an appropriately designed lowpass filter.

## c) 

* Determine the cutoff frequency $\hat{\omega}_c$ for the lowpass filter used in interpolation, and the desired filter gain `G` in the passband.

In [ ]:
w_c = "???" # Identify correct cutoff frequency
G = "???" # Identify desired passband gain 
# WRITE YOUR CODE IN THIS CELL:

In [ ]:
from hashlib import sha1
assert sha1(str(round(w_c, 5)).encode('utf-8')+b'38b4').hexdigest() == 'b121b92168b5265631b99e551bec322c2a6e6937', 'Wrong answer for w_c :('; print('Correct answer for w_c :)')
assert sha1(str(round(G, 5)).encode('utf-8')+b'2901c').hexdigest() == '33c62da8778aced26416916aac3c161d8111f0b5', 'Wrong answer for G :('; print('Correct answer for G :)')

## d)
* Use either the *window design method* or the function [remez](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.remez.html) to design a $257$-tap lowpass FIR filter with the transition region centered around $\hat{\omega}_c$ found in problem **c)**. Then, finish the interpolation process and produce a new upsampled digital signal $x_{\text{new}}[m] = x\left(m\cdot \frac{T_s}{3}\right)$. Show the magnitude spectrum of both the original signal $x[n]$, and the upsampled signal $x_{\text{new}}[m]$.<br>
*PS: Use convolution `mode="same"` to ensure output length equals input length, thereby ensuring consistent scaling of the magnitude spectrum.*

In [ ]:
# WRITE YOUR CODE IN THIS CELL:

[Here](figures/task1d_lf.png) is a plot showing what the interpolated signal magnitude spectrum should look like. 

## e)


* Study the magnitude spectrum plot for $x_{\text{new}}[m]$ and $x[n]$. Explain why our current result would be more desirable as a output of an upsampling operation thant the result in problem **a)**.


<div class="alert alert-info">
<h4> Answer theory questions here!</h4>
</div>


### Good to know:

The above process of upsampling is fairly standard, and specialized DSP libraries have functions to perform upsampling in C for STM32.
* **STM32:** The CMSIS-DSP has separte [FIR Interpolator functions](https://arm-software.github.io/CMSIS-DSP/main/group__FIR__Interpolate.html). Requires user to design a FIR filter (in Python), and store filter coeffeicients directly in the C code as an array. These functions are optimized to make use of pylyphase decimation techniques, significantly reducing the number of multiplications needed at the filtering stage of the decimator.